# 🔬 SkinCareAI: Dermatological Screening & Exploration

This notebook covers the development, training history analysis, and final model evaluation for the **SkinCareAI** dermatological screening assistant. 

### Project Objective:
Classify dermoscopy skin lesion images into 7 distinct diagnostic categories using transfer learning on **ResNet50** using the HAM10000 dataset.

---

## 1. Environment Setup & Imports
Let's import TensorFlow/Keras, standard data science libraries, and check for CPU/GPU availability.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import tensorflow as tf
from tensorflow import keras

print(f"TensorFlow Version: {tf.__version__}")
print(f"Keras Version: {keras.__version__}")
print(f"NumPy Version: {np.__version__}")

# Check GPUs
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU Detected: {gpus}")
else:
    print("No GPU detected - running on CPU.")

TensorFlow Version: 2.21.0
Keras Version: 3.15.0
NumPy Version: 2.5.1
No GPU detected - running on CPU.


## 2. Dataset Overview
We use the **HAM10000** dataset, which consists of 10,015 images. Let's load the class names and definitions.

In [2]:
CLASS_NAMES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]

CLASS_FULL_NAMES = {
    "akiec": "Actinic Keratoses / Intraepithelial Carcinoma",
    "bcc":   "Basal Cell Carcinoma",
    "bkl":   "Benign Keratosis-like Lesion",
    "df":    "Dermatofibroma",
    "mel":   "Melanoma",
    "nv":    "Melanocytic Nevi (Mole)",
    "vasc":  "Vascular Lesion",
}

print(f"Total classes: {len(CLASS_NAMES)}")
for idx, name in enumerate(CLASS_NAMES):
    print(f"  Class {idx} ({name:<5}) : {CLASS_FULL_NAMES[name]}")

Total classes: 7
  Class 0 (akiec) : Actinic Keratoses / Intraepithelial Carcinoma
  Class 1 (bcc)   : Basal Cell Carcinoma
  Class 2 (bkl)   : Benign Keratosis-like Lesion
  Class 3 (df)    : Dermatofibroma
  Class 4 (mel)   : Melanoma
  Class 5 (nv)    : Melanocytic Nevi (Mole)
  Class 6 (vasc)  : Vascular Lesion


## 3. Training Performance Analysis
Let's load the `training_log.csv` file that was generated by our training run, and plot the accuracy and loss curves.

In [3]:
log_path = 'training_log.csv'
if os.path.exists(log_path):
    df = pd.read_csv(log_path)
    print("First few training log entries:")
    print(df.head(3))
else:
    print("Log file not found!")

First few training log entries:
   epoch  accuracy  learning_rate      loss  val_accuracy  val_loss
0      0  0.315980       0.000100  3.074670      0.464339  1.425842
1      1  0.464170       0.000100  2.250314      0.594514  1.144258
2      2  0.511611       0.000100  2.011616      0.594015  1.113314


In [4]:
# Plot the accuracy and loss history
if os.path.exists(log_path):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot accuracy
    axes[0].plot(df['accuracy'], label='Train Accuracy', color='#3498db', linewidth=2)
    axes[0].plot(df['val_accuracy'], label='Val Accuracy', color='#e74c3c', linewidth=2)
    axes[0].set_title('Training & Validation Accuracy', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Epochs')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend(loc='lower right')
    axes[0].grid(alpha=0.3)

    # Plot loss
    axes[1].plot(df['loss'], label='Train Loss', color='#3498db', linewidth=2)
    axes[1].plot(df['val_loss'], label='Val Loss', color='#e74c3c', linewidth=2)
    axes[1].set_title('Training & Validation Loss', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Epochs')
    axes[1].set_ylabel('Loss')
    axes[1].legend(loc='upper right')
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("Cannot plot: log file is missing.")

## 4. Model Architecture & Custom Top Layers
Here is the Keras model definition using `ResNet50` as base model and freezing it for initial classification head tuning.

In [5]:
from src.model import build_model, print_model_info

# Construct the model
model = build_model(num_classes=len(CLASS_NAMES))
print("Base Model: ResNet50 (Frozen)")
print("Classification Head Added.")

Base Model: ResNet50 (Frozen)
Classification Head Added.


## 5. Model Evaluation & Confusion Matrix
Let's load the pre-trained weights from `models/skin_model.keras` and review the saved confusion matrix.

In [6]:
model_path = 'models/skin_model.keras'
if os.path.exists(model_path):
    model = keras.models.load_model(model_path)
    print(f"Model weights loaded successfully from {model_path}")
else:
    print("Pre-trained model file not found. Make sure to train first.")

Model weights loaded successfully from models/skin_model.keras


In [7]:
# Load and show the saved confusion matrix image
cm_path = 'confusion_matrix.png'
if os.path.exists(cm_path):
    img = Image.open(cm_path)
    plt.figure(figsize=(10, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Saved Confusion Matrix', fontsize=14, fontweight='bold')
    plt.show()
else:
    print("confusion_matrix.png not found. Try evaluating or running the train pipeline first.")

## 6. Real-time Inference Testing
Let's create a quick helper function to preprocess a test image and get diagnostic predictions.

In [8]:
from src.data_loader import preprocess_single_image
from src.model import TRIAGE_MAP

def diagnose_lesion(image_path):
    if not os.path.exists(image_path):
        print(f"Image {image_path} not found.")
        return
        
    img = Image.open(image_path).convert('RGB')
    img_arr = np.array(img)
    preprocessed = preprocess_single_image(img_arr)
    
    preds = model.predict(preprocessed, verbose=0)[0]
    pred_idx = np.argmax(preds)
    confidence = preds[pred_idx]
    pred_cls = CLASS_NAMES[pred_idx]
    
    triage_lvl, triage_color, triage_msg = TRIAGE_MAP[pred_cls]
    
    print(f"--- Diagnostic Result ---")
    print(f"Class Label : {pred_cls.upper()} ({CLASS_FULL_NAMES[pred_cls]})")
    print(f"Confidence  : {confidence*100:.2f}%")
    print(f"Triage Level: {triage_lvl} ({triage_msg})")
    print(f"-------------------------")